In [0]:
T_trans=spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_item = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_item")
T_date = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")
T_location = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")

In [0]:
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

start = "2025-10-01"
end = "2025-11-30"

PY_start = datetime.strptime(start, "%Y-%m-%d") - relativedelta(months=12)
PY_end = datetime.strptime(end, "%Y-%m-%d") - relativedelta(months=12)
CY_start = start
CY_end = end

# Trend Adjustment Periods (July of each year)
TAP_2024_start = "2024-07-01"
TAP_2024_end = "2024-07-31"
TAP_2025_start = "2025-07-01"
TAP_2025_end = "2025-07-31"
TAP_2026_start = "2026-07-01"
TAP_2026_end = "2026-07-31"

In [0]:
import pyspark.sql.functions as F

# Pre-filter T_date to only relevant date ranges (reduces join size)
T_date_filtered = (
    T_date
    .select("dateid", "datefull")
    .filter(
        (F.col("datefull").between(PY_start, PY_end)) |
        (F.col("datefull").between(CY_start, CY_end)) |
        (F.col("datefull").between(TAP_2024_start, TAP_2024_end)) |
        (F.col("datefull").between(TAP_2025_start, TAP_2025_end)) |
        (F.col("datefull").between(TAP_2026_start, TAP_2026_end))
    )
)

# Collect valid date IDs to pre-filter the large transaction table (enables Delta data skipping)
valid_date_ids = [row.dateid for row in T_date_filtered.select("dateid").collect()]

# Pre-filter and select only needed columns from T_trans
T_trans_filtered = (
    T_trans
    .filter(F.col("VisitDateId").isin(valid_date_ids))
    .filter(F.col("CDE_ID").isNotNull() & (F.col("CDE_ID") > 0))
    .select("CDE_ID", "VisitDateId", "LocationId", "ConcessionFLAG",
            "ItemSkey", "SCENEFLAG", "Quantity", "Scene_Points_Earned",
            "Scene_Points_Redeemed", "Net_Revenue", "Transaction_Number")
)

T_scene_redeemed = (
    T_trans_filtered
    .join(T_date_filtered, T_date_filtered.dateid == T_trans_filtered.VisitDateId, how='inner')
    .withColumn("Period",
                F.when(F.col("datefull").between(PY_start, PY_end), "2024")
                .when(F.col("datefull").between(CY_start, CY_end), "2025")
                .when(F.col("datefull").between(TAP_2024_start, TAP_2024_end), "2024_TAP")
                .when(F.col("datefull").between(TAP_2025_start, TAP_2025_end), "2025_TAP")
                .when(F.col("datefull").between(TAP_2026_start, TAP_2026_end), "2026_TAP"))
    .groupBy("Period", "CDE_ID", "datefull", "Transaction_Number")
    .agg(
        F.lit(1).alias("DistinctCustomers"),
        F.sum("Quantity").alias("TotalQuantity"),
        F.sum("Net_Revenue").alias("TotalRevenue"),
        F.sum("Scene_Points_Earned").alias("totalPointsEarned"),
        F.sum("Scene_Points_Redeemed").alias("totalPointsRedeemed"),
        F.max("LocationId").alias("LocationID")
    )
)


In [0]:
T_scene_redeemed_loc = (
    T_scene_redeemed
    .join(
        T_location.select("LocationID", "LC_Location_Type_Description", "LC_Trade_Brand_Description"),
        on="LocationID",
        how="inner"
    )
    .filter(F.col("LC_Location_Type_Description").like("Restaurant%"))
    .drop("LC_Location_Type_Description")
)

T_scene_redeemed_loc.display()

In [0]:
T_summary = (
    T_scene_redeemed_loc
    .withColumn("Month", F.date_format("datefull", "yyyy-MM"))
    .groupBy("Period", "Month", "LC_Trade_Brand_Description")
    .agg(
        F.countDistinct("CDE_ID").alias("DistinctCustomers"),
        F.sum("TotalRevenue").alias("TotalRevenue"),
        F.sum("totalPointsEarned").alias("TotalPointsEarned"),
        F.sum("totalPointsRedeemed").alias("TotalPointsRedeemed")
    )
    .orderBy("Period", "Month", "LC_Trade_Brand_Description")
)

T_summary.display()

In [0]:
from pyspark.sql.window import Window

# Aggregate total points redeemed per customer per brand
T_customer_points = (
    T_scene_redeemed_loc
    .groupBy("Period", "LC_Trade_Brand_Description", "CDE_ID")
    .agg(F.sum("totalPointsRedeemed").alias("TotalPointsRedeemed"))
)

# Show redeemer vs non-redeemer split by brand for context
T_redeem_summary = (
    T_customer_points
    .withColumn("Redeemed", F.when(F.col("TotalPointsRedeemed") > 0, "Yes").otherwise("No"))
    .groupBy("Period", "LC_Trade_Brand_Description", "Redeemed")
    .agg(F.count("CDE_ID").alias("CustomerCount"))
    .orderBy("Period", "LC_Trade_Brand_Description", "Redeemed")
)
print("=== Redeemer vs Non-Redeemer Split by Brand ===")
T_redeem_summary.display()

# Percentile analysis on customers who actually redeemed (> 0 points)
T_redeemers = T_customer_points.filter(F.col("TotalPointsRedeemed") > 0)

w = Window.partitionBy("Period", "LC_Trade_Brand_Description").orderBy("TotalPointsRedeemed")

T_percentiles = (
    T_redeemers
    .withColumn("Decile", F.ntile(10).over(w))
    .groupBy("Period", "LC_Trade_Brand_Description", "Decile")
    .agg(
        F.count("CDE_ID").alias("CustomerCount"),
        F.min("TotalPointsRedeemed").alias("MinPointsRedeemed"),
        F.max("TotalPointsRedeemed").alias("MaxPointsRedeemed"),
        F.round(F.avg("TotalPointsRedeemed"), 2).alias("AvgPointsRedeemed"),
        F.sum("TotalPointsRedeemed").alias("TotalPointsRedeemed")
    )
    .withColumn("PercentileLabel", F.concat(F.lit("Top "), ((F.lit(11) - F.col("Decile")) * 10).cast("int"), F.lit("%")))
    .orderBy("Period", "LC_Trade_Brand_Description", F.desc("Decile"))
)

print("\n=== Percentile Distribution (Redeemers Only) ===")
T_percentiles.display()

In [0]:
# Build a broader base: ALL transactions at Restaurant locations (including non-Scene members)
T_all_trans = (
    T_trans
    .filter(F.col("VisitDateId").isin(valid_date_ids))
    .select("CDE_ID", "VisitDateId", "LocationId", "Transaction_Number",
            "Scene_Points_Earned", "Scene_Points_Redeemed")
    .join(T_date_filtered, T_date_filtered.dateid == F.col("VisitDateId"), how="inner")
    .join(
        T_location.select("LocationID", "LC_Location_Type_Description", "LC_Trade_Brand_Description"),
        on="LocationID",
        how="inner"
    )
    .filter(F.col("LC_Location_Type_Description").like("Restaurant%"))
    .withColumn("Period",
                F.when(F.col("datefull").between(PY_start, PY_end), "2024")
                .when(F.col("datefull").between(CY_start, CY_end), "2025")
                .when(F.col("datefull").between(TAP_2024_start, TAP_2024_end), "2024_TAP")
                .when(F.col("datefull").between(TAP_2025_start, TAP_2025_end), "2025_TAP")
                .when(F.col("datefull").between(TAP_2026_start, TAP_2026_end), "2026_TAP"))
)

# High-level overview by Period and Brand
T_overview = (
    T_all_trans
    .groupBy("Period", "LC_Trade_Brand_Description")
    .agg(
        F.countDistinct("Transaction_Number").alias("TotalDistinctTransactions"),
        F.countDistinct(
            F.when((F.col("CDE_ID").isNotNull()) & (F.col("CDE_ID") > 0), F.col("Transaction_Number"))
        ).alias("SceneTransactions"),
        F.countDistinct(
            F.when((F.col("CDE_ID").isNotNull()) & (F.col("CDE_ID") > 0), F.col("CDE_ID"))
        ).alias("DistinctSceneCustomers"),
        F.countDistinct(
            F.when((F.col("CDE_ID").isNotNull()) & (F.col("CDE_ID") > 0) & (F.col("Scene_Points_Redeemed") > 0), F.col("CDE_ID"))
        ).alias("CustomersWhoRedeemed")
    )
    .withColumn("CustomersWhoDidNotRedeem", F.col("DistinctSceneCustomers") - F.col("CustomersWhoRedeemed"))
    .withColumn("NonSceneTransactions", F.col("TotalDistinctTransactions") - F.col("SceneTransactions"))
    .withColumn("SceneTransactionPct",
        F.when(F.col("TotalDistinctTransactions") > 0,
               F.round(F.col("SceneTransactions") / F.col("TotalDistinctTransactions") * 100, 2))
    )
    .withColumn("RedemptionRate",
        F.when(F.col("DistinctSceneCustomers") > 0,
               F.round(F.col("CustomersWhoRedeemed") / F.col("DistinctSceneCustomers") * 100, 2))
    )
    .orderBy("Period", "LC_Trade_Brand_Description")
)

print("=== Overview ===")
T_overview.display()

# Per-visit redemption rate for customers who redeem
# Step 1: Get visit-level data for Scene members, deduplicating shared Transaction_Numbers
# Assign each transaction to one CDE_ID (highest points redeemed wins, then highest CDE_ID as tiebreaker)
from pyspark.sql.window import Window

w_dedup = Window.partitionBy("Period", "LC_Trade_Brand_Description", "Transaction_Number").orderBy(
    F.desc("PointsRedeemed"), F.desc("CDE_ID")
)

T_scene_visits = (
    T_all_trans
    .filter((F.col("CDE_ID").isNotNull()) & (F.col("CDE_ID") > 0))
    .groupBy("Period", "LC_Trade_Brand_Description", "CDE_ID", "Transaction_Number")
    .agg(F.sum("Scene_Points_Redeemed").alias("PointsRedeemed"))
    .withColumn("_rank", F.row_number().over(w_dedup))
    .filter(F.col("_rank") == 1)
    .drop("_rank")
    .withColumn("VisitHadRedemption", F.when(F.col("PointsRedeemed") > 0, 1).otherwise(0))
)

# Step 2: Aggregate per customer — total visits vs visits with redemption
T_customer_visits = (
    T_scene_visits
    .groupBy("Period", "LC_Trade_Brand_Description", "CDE_ID")
    .agg(
        F.count("Transaction_Number").alias("TotalVisits"),
        F.sum("VisitHadRedemption").alias("VisitsWithRedemption")
    )
    .withColumn("IsRedeemer", F.when(F.col("VisitsWithRedemption") > 0, "Yes").otherwise("No"))
)

# Step 3: Summarize — for redeemers, what % of their visits involve a redemption?
T_visit_redemption = (
    T_customer_visits
    .groupBy("Period", "LC_Trade_Brand_Description", "IsRedeemer")
    .agg(
        F.count("CDE_ID").alias("CustomerCount"),
        F.sum("TotalVisits").alias("TotalVisits"),
        F.sum("VisitsWithRedemption").alias("VisitsWithRedemption"),
        F.round(F.avg("TotalVisits"), 2).alias("AvgVisitsPerCustomer"),
        F.round(F.avg("VisitsWithRedemption"), 2).alias("AvgRedemptionVisitsPerCustomer")
    )
    .withColumn("PerVisitRedemptionRate",
        F.when(F.col("TotalVisits") > 0,
               F.round(F.col("VisitsWithRedemption") / F.col("TotalVisits") * 100, 2))
    )
    .orderBy("Period", "LC_Trade_Brand_Description", "IsRedeemer")
)

print("\n=== Per-Visit Redemption Rate (Redeemers vs Non-Redeemers) ===")
T_visit_redemption.display()

In [0]:
# ============================================================
# YoY Redemption Pattern Comparison & 2026 Oct-Nov Projection
# Uses TAP (July) trend as a leading indicator to adjust forecasts
# ============================================================

from pyspark.sql.functions import col, sum as _sum, countDistinct, round as _round, lit

# --- 1. Aggregate key metrics by Period and Brand ---
T_period_metrics = (
    T_scene_redeemed_loc
    .groupBy("Period", "LC_Trade_Brand_Description")
    .agg(
        countDistinct("CDE_ID").alias("DistinctCustomers"),
        countDistinct("Transaction_Number").alias("DistinctTransactions"),
        _sum("TotalRevenue").alias("TotalRevenue"),
        _sum("totalPointsEarned").alias("TotalPointsEarned"),
        _sum("totalPointsRedeemed").alias("TotalPointsRedeemed"),
        _sum("TotalQuantity").alias("TotalQuantity")
    )
    .withColumn("RevenuePerCustomer", _round(col("TotalRevenue") / col("DistinctCustomers"), 2))
    .withColumn("PointsRedeemedPerCustomer", _round(col("TotalPointsRedeemed") / col("DistinctCustomers"), 2))
    .withColumn("TransactionsPerCustomer", _round(col("DistinctTransactions") / col("DistinctCustomers"), 2))
)

T_period_metrics_pd = T_period_metrics.toPandas()

# --- 2. Pivot for YoY comparison ---
import pandas as pd

metrics_pivot = T_period_metrics_pd.pivot_table(
    index="LC_Trade_Brand_Description",
    columns="Period",
    values=["DistinctCustomers", "DistinctTransactions", "TotalRevenue",
            "TotalPointsRedeemed", "RevenuePerCustomer", "PointsRedeemedPerCustomer", "TransactionsPerCustomer"],
    aggfunc="first"
)

# --- 3. Calculate YoY growth rates ---
brands = T_period_metrics_pd["LC_Trade_Brand_Description"].unique()
core_periods = ["2024", "2025"]
tap_periods = ["2024_TAP", "2025_TAP", "2026_TAP"]

results = []
for brand in brands:
    brand_data = T_period_metrics_pd[T_period_metrics_pd["LC_Trade_Brand_Description"] == brand]
    
    row = {"Brand": brand}
    
    # Core period metrics (Oct-Nov)
    for metric in ["DistinctCustomers", "TotalRevenue", "TotalPointsRedeemed", "DistinctTransactions",
                   "RevenuePerCustomer", "PointsRedeemedPerCustomer"]:
        val_2024 = brand_data.loc[brand_data["Period"] == "2024", metric].values
        val_2025 = brand_data.loc[brand_data["Period"] == "2025", metric].values
        
        val_2024 = float(val_2024[0]) if len(val_2024) > 0 else 0
        val_2025 = float(val_2025[0]) if len(val_2025) > 0 else 0
        
        row[f"{metric}_2024"] = val_2024
        row[f"{metric}_2025"] = val_2025
        row[f"{metric}_YoY_Growth"] = ((val_2025 - val_2024) / val_2024 * 100) if val_2024 != 0 else None
    
    # TAP metrics (July) for trend projection
    for metric in ["DistinctCustomers", "TotalRevenue", "TotalPointsRedeemed", "DistinctTransactions"]:
        tap_24 = brand_data.loc[brand_data["Period"] == "2024_TAP", metric].values
        tap_25 = brand_data.loc[brand_data["Period"] == "2025_TAP", metric].values
        tap_26 = brand_data.loc[brand_data["Period"] == "2026_TAP", metric].values
        
        tap_24 = float(tap_24[0]) if len(tap_24) > 0 else 0
        tap_25 = float(tap_25[0]) if len(tap_25) > 0 else 0
        tap_26 = float(tap_26[0]) if len(tap_26) > 0 else 0
        
        row[f"{metric}_TAP24"] = tap_24
        row[f"{metric}_TAP25"] = tap_25
        row[f"{metric}_TAP26"] = tap_26
        
        # TAP growth rates
        row[f"{metric}_TAP_24to25_Growth"] = ((tap_25 - tap_24) / tap_24 * 100) if tap_24 != 0 else None
        row[f"{metric}_TAP_25to26_Growth"] = ((tap_26 - tap_25) / tap_25 * 100) if tap_25 != 0 else None
        
        # --- PROJECTION: Oct-Nov 2026 ---
        # Method: Apply TAP 2025→2026 growth rate to 2025 actuals
        val_2025 = row.get(f"{metric}_2025", 0)
        tap_growth_25to26 = ((tap_26 - tap_25) / tap_25) if tap_25 != 0 else 0
        row[f"{metric}_2026_Projected"] = round(val_2025 * (1 + tap_growth_25to26), 0)
    
    results.append(row)

df_analysis = pd.DataFrame(results)

# --- 4. Display YoY Comparison ---
print("=" * 80)
print("YoY REDEMPTION PATTERN COMPARISON (Oct-Nov: 2024 vs 2025)")
print("=" * 80)
yoy_cols = ["Brand",
            "DistinctCustomers_2024", "DistinctCustomers_2025", "DistinctCustomers_YoY_Growth",
            "TotalRevenue_2024", "TotalRevenue_2025", "TotalRevenue_YoY_Growth",
            "TotalPointsRedeemed_2024", "TotalPointsRedeemed_2025", "TotalPointsRedeemed_YoY_Growth",
            "RevenuePerCustomer_2024", "RevenuePerCustomer_2025", "RevenuePerCustomer_YoY_Growth",
            "PointsRedeemedPerCustomer_2024", "PointsRedeemedPerCustomer_2025", "PointsRedeemedPerCustomer_YoY_Growth"]
df_yoy = df_analysis[yoy_cols].copy()
df_yoy.columns = [c.replace("_YoY_Growth", "_%chg") for c in df_yoy.columns]
display(spark.createDataFrame(df_yoy))

# --- 5. Display TAP Trend (July Trend Indicator) ---
print("\n" + "=" * 80)
print("TAP TREND (July as Leading Indicator: 2024 → 2025 → 2026)")
print("=" * 80)
tap_cols = ["Brand",
            "DistinctCustomers_TAP24", "DistinctCustomers_TAP25", "DistinctCustomers_TAP26",
            "DistinctCustomers_TAP_24to25_Growth", "DistinctCustomers_TAP_25to26_Growth",
            "TotalRevenue_TAP24", "TotalRevenue_TAP25", "TotalRevenue_TAP26",
            "TotalRevenue_TAP_24to25_Growth", "TotalRevenue_TAP_25to26_Growth",
            "TotalPointsRedeemed_TAP24", "TotalPointsRedeemed_TAP25", "TotalPointsRedeemed_TAP26",
            "TotalPointsRedeemed_TAP_24to25_Growth", "TotalPointsRedeemed_TAP_25to26_Growth"]
display(spark.createDataFrame(df_analysis[tap_cols]))

# --- 6. Display 2026 Oct-Nov Projection ---
print("\n" + "=" * 80)
print("PROJECTED OCT-NOV 2026 (Based on TAP 2025→2026 Growth Applied to 2025 Actuals)")
print("=" * 80)
proj_cols = ["Brand",
             "DistinctCustomers_2025", "DistinctCustomers_2026_Projected",
             "TotalRevenue_2025", "TotalRevenue_2026_Projected",
             "TotalPointsRedeemed_2025", "TotalPointsRedeemed_2026_Projected",
             "DistinctTransactions_2025", "DistinctTransactions_2026_Projected"]
df_proj = df_analysis[proj_cols].copy()

# Add growth columns for clarity
df_proj["Customers_Proj_Growth_%"] = ((df_proj["DistinctCustomers_2026_Projected"] - df_proj["DistinctCustomers_2025"]) / df_proj["DistinctCustomers_2025"] * 100).round(1)
df_proj["Revenue_Proj_Growth_%"] = ((df_proj["TotalRevenue_2026_Projected"] - df_proj["TotalRevenue_2025"]) / df_proj["TotalRevenue_2025"] * 100).round(1)
df_proj["PointsRedeemed_Proj_Growth_%"] = ((df_proj["TotalPointsRedeemed_2026_Projected"] - df_proj["TotalPointsRedeemed_2025"]) / df_proj["TotalPointsRedeemed_2025"] * 100).round(1)

display(spark.createDataFrame(df_proj))

# --- 7. Summary narrative ---
print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
for _, r in df_analysis.iterrows():
    brand = r["Brand"]
    cust_yoy = r.get("DistinctCustomers_YoY_Growth", 0) or 0
    rev_yoy = r.get("TotalRevenue_YoY_Growth", 0) or 0
    pts_yoy = r.get("TotalPointsRedeemed_YoY_Growth", 0) or 0
    
    tap_cust_growth = r.get("DistinctCustomers_TAP_25to26_Growth", 0) or 0
    tap_rev_growth = r.get("TotalRevenue_TAP_25to26_Growth", 0) or 0
    tap_pts_growth = r.get("TotalPointsRedeemed_TAP_25to26_Growth", 0) or 0
    
    print(f"\n{brand}:")
    print(f"  Oct-Nov YoY (2024→2025): Customers {cust_yoy:+.1f}%, Revenue {rev_yoy:+.1f}%, Points Redeemed {pts_yoy:+.1f}%")
    print(f"  TAP Trend (Jul 2025→2026):  Customers {tap_cust_growth:+.1f}%, Revenue {tap_rev_growth:+.1f}%, Points Redeemed {tap_pts_growth:+.1f}%")
    print(f"  → 2026 Oct-Nov Projection: ~{int(r.get('DistinctCustomers_2026_Projected', 0)):,} customers, "
          f"${int(r.get('TotalRevenue_2026_Projected', 0)):,} revenue, "
          f"{int(r.get('TotalPointsRedeemed_2026_Projected', 0)):,} points redeemed")

In [0]:
# ============================================================
# GAMEBAND FREE-GAME REWARD — BREAK-EVEN DESIGN
# Offer: customers redeem points to buy gamebands; free games sweeten
#        bigger bands to drive upgrades. Solve for profitable reward levels.
#
# Value:  each redeemed point is worth $0.00942 to the business
# Cost:   each free game = up to $7 opportunity cost (assume every game a lost sale)
# Reward: free games per band, 0..15 (15 = absolute max, top band)
#
# Model 1: min points a customer must redeem for a reward to break even
# Model 2: if points don't cover it, the incremental CASH spend an upgrade
#          must drive to break even (no assumed number — solved)
# ============================================================

POINT_VALUE = 0.00942          # business value per redeemed point
GAME_COST   = 7.0              # max opportunity cost per free game
MAX_GAMES   = 15
BANDS       = [15, 25, 35, 50, 75, 100, 129]   # gameband prices ($)
PTS_PER_DOLLAR = 100           # 100 pts = $1 face value (to size band in points)

# ---------- MODEL 1: min points threshold to break even per reward level ----------
# Break-even: points_redeemed * POINT_VALUE >= free_games * GAME_COST
m1_rows = []
for g in range(1, MAX_GAMES + 1):
    tmin = g * GAME_COST / POINT_VALUE
    m1_rows.append({
        "Free_Games": g,
        "Opp_Cost": round(g * GAME_COST, 2),
        "BreakEven_Points_Redeemed": round(tmin),
    })
m1 = pd.DataFrame(m1_rows)

print("=" * 80)
print("MODEL 1 — Min points a customer must redeem for the reward to break even")
print("=" * 80)
display(spark.createDataFrame(m1))

# ---------- Per-band recommended reward, anchored to real redemption ----------
# For each band, the customer redeems ~band_price worth of points to buy it
# (band_price * 100 pts). Max free games that band's redemption can justify:
#   max_games = floor( band_points * POINT_VALUE / GAME_COST )
print("\n" + "=" * 80)
print("RECOMMENDED MAX FREE GAMES PER BAND (profitable on redemption alone)")
print("=" * 80)
band_rows = []
for b in BANDS:
    band_points = b * PTS_PER_DOLLAR                      # pts to cover the band
    max_g = int((band_points * POINT_VALUE) / GAME_COST)  # floor
    max_g = min(max_g, MAX_GAMES)
    band_rows.append({
        "Band_$": b,
        "Points_To_Redeem": band_points,
        "Max_Free_Games_BreakEven": max_g,
        "Reward_Opp_Cost": round(max_g * GAME_COST, 2),
        "Points_Value": round(band_points * POINT_VALUE, 2),
    })
m_band = pd.DataFrame(band_rows)
display(spark.createDataFrame(m_band))

# ---------- MODEL 2: break-even incremental cash spend per upgrade ----------
# Customer upgrades from band i -> band i+1, granted g free games for the bigger band.
# Extra points redeemed to cover the bigger band = band_jump * 100, worth
# band_jump*100*POINT_VALUE to the business. Any remaining gap to cover the
# free-game cost must come from incremental CASH spend:
#   incr_spend_needed = g*GAME_COST - (band_jump*100*POINT_VALUE)
print("\n" + "=" * 80)
print("MODEL 2 — Break-even incremental CASH spend per upgrade")
print("(<=0 means already profitable on points alone; >0 = cash the upgrade must drive)")
print("=" * 80)
m2_rows = []
for i in range(len(BANDS) - 1):
    frm, to = BANDS[i], BANDS[i + 1]
    band_jump   = to - frm
    extra_pts   = band_jump * PTS_PER_DOLLAR
    pts_value   = extra_pts * POINT_VALUE
    row = {"Upgrade": f"${frm}->${to}"}
    for g in [1, 3, 5, 10, 15]:
        row[f"{g}g"] = round(g * GAME_COST - pts_value, 1)
    m2_rows.append(row)
m2 = pd.DataFrame(m2_rows)
display(spark.createDataFrame(m2))

print("\n" + "-" * 80)
print("READ:")
print("  Model 1: pick each band's reward so BreakEven_Points <= what that")
print("           band's customers actually redeem. Median redeemer ~3,000 pts")
print("           supports ~4 free games profitably; 15 games needs ~11,150 pts.")
print("  Model 2: small rewards (1 game) are free money (negative = no cash needed).")
print("           Big rewards (10-15 games) need real cash upgrades to justify —")
print("           reserve those for top bands where top-up is plausible.")
print("-" * 80)

In [0]:
# ============================================================
# REDEEMER SEGMENTATION — group redeemers by Scene points redeemed
# Behavioural distribution only (2025, projected to 2026). No cost/games yet.
# Sources: T_scene_redeemed_loc (c3), df_analysis (c7)
# ============================================================

import numpy as np
# --- 1. 2025 redeemer distribution ---
_r25 = (
    T_scene_redeemed_loc.filter(F.col("Period") == "2025")
    .groupBy("LC_Trade_Brand_Description", "CDE_ID")
    .agg(F.sum("totalPointsRedeemed").alias("Pts")).toPandas()
)
_r25["Pts"] = _r25["Pts"].astype(float)
_r25 = _r25[_r25["Pts"] > 0]

# --- 2. Per-brand 2026 growth factor (cell 7) ---
growth_factor = {}
for _, r in df_analysis.iterrows():
    c25 = float(r.get("DistinctCustomers_2025", 0) or 0)
    c26 = float(r.get("DistinctCustomers_2026_Projected", 0) or 0)
    growth_factor[r["Brand"]] = (c26 / c25) if c25 else 1.0

# --- 3. Point buckets (even ranges) ---
bucket_edges = [0, 1400,2500,3500,4500,5500,6500,7500,8500,9500,10500, np.inf]
bucket_labels = ["0-1.5k", "1.5k-2.5k", "2.5k-3.5k", "3.5k-4.5k", "4.5k-5.5k", "5.5k-6.5k", "6.5k-7.5k", "7.5k-8.5k", "8.5k-9.5k", "9.5k-10.5k", ">10.5k"]

# --- 4. Count redeemers per bucket, per brand and combined, projected to 2026 ---
rows = []
for i in range(len(bucket_labels)):
    lo, hi = bucket_edges[i], bucket_edges[i + 1]
    row = {"Points_Bucket": bucket_labels[i]}
    combined = 0.0
    for brand, gf in growth_factor.items():
        bd = _r25[_r25["LC_Trade_Brand_Description"] == brand]
        n = ((bd["Pts"] >= lo) & (bd["Pts"] < hi)).sum() * gf
        row[brand] = round(n)
        combined += n
    row["Combined"] = round(combined)
    rows.append(row)

seg = pd.DataFrame(rows)
# add % of total
tot = seg["Combined"].sum()
seg["Pct_of_Redeemers"] = (seg["Combined"] / tot * 100).round(1)

print("REDEEMER SEGMENTATION by Scene points redeemed (2026 projected)")
display(spark.createDataFrame(seg))

ITERATION 2

In [0]:
# ============================================================
# DATA AVAILABILITY CHECK — confirm continuous monthly coverage
# before widening the window. Reads only; changes nothing.
# Target trend-fit window: Aug 2024 -> Jul 2026 (24 months)
# ============================================================
import pyspark.sql.functions as F

TREND_START = "2024-08-01"
TREND_END   = "2026-07-31"   # Jul 2026 = latest full month

_avail = (
    T_trans
    .filter(F.col("CDE_ID").isNotNull() & (F.col("CDE_ID") > 0))
    .join(T_date.select("dateid", "datefull"),
          T_date.dateid == T_trans.VisitDateId, how="inner")
    .join(T_location.select("LocationID", "LC_Location_Type_Description", "LC_Trade_Brand_Description"),
          on="LocationID", how="inner")
    .filter(F.col("LC_Location_Type_Description").like("Restaurant%"))
    .filter(F.col("datefull").between(TREND_START, TREND_END))
    .withColumn("Month", F.date_format("datefull", "yyyy-MM"))
    .groupBy("Month", "LC_Trade_Brand_Description")
    .agg(
        F.countDistinct("CDE_ID").alias("DistinctMembers"),
        F.countDistinct(
            F.when(F.col("Scene_Points_Redeemed") > 0, F.col("CDE_ID"))
        ).alias("Redeemers"),
        F.countDistinct("Transaction_Number").alias("Visits"),
        F.sum("Net_Revenue").alias("Revenue"),
        F.sum("Scene_Points_Redeemed").alias("PointsRedeemed"),
    )
    .orderBy("Month", "LC_Trade_Brand_Description")
)
_avail_pd = _avail.toPandas()

# --- Gate 1: continuous months, no gaps ---
import pandas as pd
expected = pd.period_range("2024-08", "2026-07", freq="M").astype(str).tolist()
got = sorted(_avail_pd["Month"].unique())
missing = [m for m in expected if m not in got]
print(f"Expected {len(expected)} months (Aug-2024..Jul-2026)")
print(f"Found {len(got)} months. Missing: {missing if missing else 'none'}")

# --- Gate 2: both brands present every month ---
brands = _avail_pd["LC_Trade_Brand_Description"].unique().tolist()
print(f"Brands present: {brands}")
pivot_n = _avail_pd.pivot_table(index="Month", columns="LC_Trade_Brand_Description",
                                values="DistinctMembers", aggfunc="first")
print("\nMonths where a brand is missing/zero members:")
print(pivot_n[pivot_n.isna().any(axis=1)] if pivot_n.isna().any().any() else "  none")

# --- Gate 3: eyeball the redemption rate series + flag July ---
_avail_pd["RedemptionRate"] = (_avail_pd["Redeemers"] / _avail_pd["DistinctMembers"] * 100).round(2)
print("\n=== Monthly redemption rate by brand (%) ===")
display(spark.createDataFrame(
    _avail_pd.pivot_table(index="Month", columns="LC_Trade_Brand_Description",
                          values="RedemptionRate", aggfunc="first").reset_index()
))

# --- July anomaly pre-check: each July vs the series ---
print("\n=== July check: rate in each July vs 24-month brand mean/std ===")
for b in brands:
    bd = _avail_pd[_avail_pd["LC_Trade_Brand_Description"] == b]
    mu, sd = bd["RedemptionRate"].mean(), bd["RedemptionRate"].std()
    julys = bd[bd["Month"].str.endswith("-07")][["Month", "RedemptionRate"]]
    print(f"\n{b}: 24mo mean {mu:.2f}% ± {sd:.2f}")
    for _, r in julys.iterrows():
        z = (r["RedemptionRate"] - mu) / sd if sd else 0
        flag = "  <-- outlier" if abs(z) > 1.5 else ""
        print(f"  {r['Month']}: {r['RedemptionRate']:.2f}%  (z={z:+.2f}){flag}")

In [0]:
# ============================================================
# T_monthly — continuous monthly grain for redemption trend fit
# Independent of the Period pipeline (cells 1-2 untouched).
# Window: Aug 2024 -> Jul 2026 (24 full months). Restaurant locations.
# Dedup grain matches cell 2: (Month, CDE_ID, Transaction_Number)
# so a shared transaction isn't double-counted.
# ============================================================
import pyspark.sql.functions as F

MONTHLY_START, MONTHLY_END = "2024-08-01", "2026-07-31"

# 1. one row per (customer, transaction, month) — preserves cell-2 dedup grain
_visits = (
    T_trans
    .filter(F.col("CDE_ID").isNotNull() & (F.col("CDE_ID") > 0))
    .join(T_date.select("dateid", "datefull"),
          T_date.dateid == T_trans.VisitDateId, how="inner")
    .filter(F.col("datefull").between(MONTHLY_START, MONTHLY_END))
    .join(T_location.select("LocationID", "LC_Location_Type_Description", "LC_Trade_Brand_Description"),
          on="LocationID", how="inner")
    .filter(F.col("LC_Location_Type_Description").like("Restaurant%"))
    .withColumn("Month", F.date_format("datefull", "yyyy-MM"))
    .groupBy("Month", "LC_Trade_Brand_Description", "CDE_ID", "Transaction_Number")
    .agg(F.sum("Scene_Points_Redeemed").alias("TxnPointsRedeemed"))
    .withColumn("TxnRedeemed", F.when(F.col("TxnPointsRedeemed") > 0, 1).otherwise(0))
)

# 2. roll up to Month x brand
T_monthly = (
    _visits
    .groupBy("Month", "LC_Trade_Brand_Description")
    .agg(
        F.countDistinct("CDE_ID").alias("DistinctMembers"),
        F.countDistinct(
            F.when(F.col("TxnPointsRedeemed") > 0, F.col("CDE_ID"))
        ).alias("Redeemers"),
        F.countDistinct("Transaction_Number").alias("Visits"),
        F.countDistinct(
            F.when(F.col("TxnRedeemed") == 1, F.col("Transaction_Number"))
        ).alias("RedeemerVisits"),
        F.sum("TxnPointsRedeemed").alias("PointsRedeemed"),
    )
)

# --- Gate: reproduce a known month from the detection run (Rec Room 2025-10 = 7.42%) ---
_chk = T_monthly.toPandas()
_rr = _chk[(_chk["LC_Trade_Brand_Description"]=="Rec Room") & (_chk["Month"]=="2025-10")].iloc[0]
_rate = _rr["Redeemers"]/_rr["DistinctMembers"]*100
print(f"Check Rec Room 2025-10 redemption rate: {_rate:.2f}% (expected ~7.42)")
assert abs(_rate - 7.42) < 0.15, "monthly grain does not reproduce detection-run rate"
print(f"Rows: {len(_chk)} (expect 48 = 24 months x 2 brands)")
assert len(_chk) == 48, "expected 24 continuous months x 2 brands"
print("T_monthly ✓")

In [0]:
# ============================================================
# REDEMPTION PROJECTION — Oct & Nov 2026, per brand
# Method: full-24mo OLS trend + calendar-month seasonal factor,
#         fit independently on each behavioural driver.
#   1. redemption_rate  = redeemers / distinct_members
#   2. visits_per_redeemer
#   3. points_per_redeemer
# Composed: projected_redeemers = proj_rate * projected_member_base
# Depends on: T_monthly (Month, brand, DistinctMembers, Redeemers,
#             Visits, PointsRedeemed, RedeemerVisits) from widened cell 2
# ============================================================
import numpy as np
import pandas as pd

TREND_START, TREND_END = "2024-08", "2026-07"          # 24-month fit window
TARGETS = {"Oct-2026": ("10", 26), "Nov-2026": ("11", 27)}  # (calendar, month index)

m = T_monthly.toPandas().copy()
m = m[(m["Month"] >= TREND_START) & (m["Month"] <= TREND_END)].sort_values("Month")

# per-redeemer behavioural metrics
m["RedemptionRate"]     = m["Redeemers"]    / m["DistinctMembers"]
m["VisitsPerRedeemer"]  = m["RedeemerVisits"] / m["Redeemers"]
m["PointsPerRedeemer"]  = m["PointsRedeemed"] / m["Redeemers"]

def fit_project(sub, ycol):
    """full-24mo OLS trend + per-calendar-month seasonal factor."""
    sub = sub.sort_values("Month").reset_index(drop=True)
    idx = np.arange(len(sub))
    cal = sub["Month"].str[-2:].values
    y   = sub[ycol].astype(float).values
    b1, b0 = np.polyfit(idx, y, 1)
    resid  = y - (b0 + b1*idx)
    season = {c: resid[cal == c].mean() for c in np.unique(cal)}
    return {lbl: b0 + b1*mi + season.get(c, 0.0) for lbl,(c,mi) in TARGETS.items()}, b1

# --- project member base (denominator) the same way, to compose counts ---
rows = []
for brand, sub in m.groupby("LC_Trade_Brand_Description"):
    rate_p,  rate_slope  = fit_project(sub, "RedemptionRate")
    memb_p,  _           = fit_project(sub, "DistinctMembers")
    vis_p,   _           = fit_project(sub, "VisitsPerRedeemer")
    pts_p,   _           = fit_project(sub, "PointsPerRedeemer")
    for lbl in TARGETS:
        proj_members  = max(memb_p[lbl], 0)
        proj_rate     = min(max(rate_p[lbl], 0), 1)          # clip to [0,1]
        proj_redeemers = proj_rate * proj_members
        proj_total_pts = proj_redeemers * pts_p[lbl]
        rows.append({
            "Brand": brand, "Period": lbl,
            "Proj_RedemptionRate_%": round(proj_rate*100, 2),
            "Proj_DistinctMembers":  round(proj_members),
            "Proj_Redeemers":        round(proj_redeemers),
            "Proj_VisitsPerRedeemer": round(vis_p[lbl], 2),
            "Proj_PointsPerRedeemer": round(pts_p[lbl]),
            "Proj_TotalPointsRedeemed": round(proj_total_pts),
            "RateTrend_pp_per_yr":   round(rate_slope*12*100, 2),
        })

df_redemption_proj = pd.DataFrame(rows).sort_values(["Brand", "Period"])

# --- VERIFICATION GATES (fail loudly) ---
_hist = m[m["Month"].str[-2:].isin(["10","11"])]
for _, r in df_redemption_proj.iterrows():
    b, p = r["Brand"], r["Period"]
    jul = m[(m["LC_Trade_Brand_Description"]==b) & (m["Month"]=="2026-07")]["RedemptionRate"].iloc[0]*100
    hist = _hist[_hist["LC_Trade_Brand_Description"]==b]["RedemptionRate"].mean()*100
    lo, hi = sorted([jul, hist])
    assert lo-1.5 <= r["Proj_RedemptionRate_%"] <= hi+1.5, \
        f"{b} {p} rate {r['Proj_RedemptionRate_%']} outside [{lo:.1f},{hi:.1f}] band"

print("=== Projected Redemption Behaviour — Oct & Nov 2026 ===")
display(spark.createDataFrame(df_redemption_proj))
print("\nAll projections within seasonal sanity bounds ✓")

In [0]:
# ============================================================
# REDEEMER BUCKET BREAKDOWN — Oct AND Nov 2026 projection
# Shape: 2025 per-customer points-redeemed distribution (real rows)
# Scale: each brand x month -> Proj_Redeemers from df_redemption_proj
# NOTE: months are independent views; redeemers do NOT sum across
#       months (a person active in both is counted in each).
# Depends on: T_scene_redeemed_loc (2025 rows), df_redemption_proj
# ============================================================
import numpy as np, pandas as pd

MONTHS = ["Oct-2026", "Nov-2026"]

# --- 1. Projected redeemers per brand x month (scale targets) ---
_p = df_redemption_proj[df_redemption_proj["Period"].isin(MONTHS)]
proj_redeemers = {(r["Brand"], r["Period"]): r["Proj_Redeemers"] for _, r in _p.iterrows()}
print("Projected redeemers:", proj_redeemers)

# --- 2. 2025 redeemer distribution (the SHAPE) ---
_r25 = (
    T_scene_redeemed_loc.filter(F.col("Period") == "2025")
    .groupBy("LC_Trade_Brand_Description", "CDE_ID")
    .agg(F.sum("totalPointsRedeemed").alias("Pts")).toPandas()
)
_r25["Pts"] = _r25["Pts"].astype(float)
_r25 = _r25[_r25["Pts"] > 0]

# --- 3. Buckets ---
edges  = [0,1400,2500,3500,4500,5500,6500,7500,8500,9500,10500, np.inf]
labels = ["0-1.4k","1.4k-2.5k","2.5k-3.5k","3.5k-4.5k","4.5k-5.5k","5.5k-6.5k",
          "6.5k-7.5k","7.5k-8.5k","8.5k-9.5k","9.5k-10.5k",">10.5k"]
_r25["Bucket"] = pd.cut(_r25["Pts"], bins=edges, labels=labels, right=False)

# --- 4. Per-brand share by bucket (computed once, reused per month) ---
shares = {}
for brand in ["Rec Room", "Playdium"]:
    s = _r25[_r25["LC_Trade_Brand_Description"] == brand].groupby("Bucket", observed=False).size()
    shares[brand] = s / s.sum()

# --- 5. Scale to each brand x month ---
out = pd.DataFrame({"Points_Bucket": labels})
for month in MONTHS:
    mtag = month.split("-")[0]  # "Oct" / "Nov"
    for brand in ["Rec Room", "Playdium"]:
        scaled = (shares[brand] * proj_redeemers[(brand, month)]).round().astype(int)
        out[f"{brand} ({mtag})"] = out["Points_Bucket"].map(scaled).fillna(0).astype(int)
    out[f"Total ({mtag})"] = out[f"Rec Room ({mtag})"] + out[f"Playdium ({mtag})"]
    tot = out[f"Total ({mtag})"].sum()
    out[f"Share ({mtag})"] = (out[f"Total ({mtag})"]/tot*100).round(0).astype(int).astype(str)+"%"

# --- Gate: each brand x month reconciles to its projection ---
for (brand, month), target in proj_redeemers.items():
    mtag = month.split("-")[0]
    got = out[f"{brand} ({mtag})"].sum()
    assert abs(got - round(target)) <= len(labels), \
        f"{brand} {month} buckets ({got}) don't reconcile to projection ({target})"

for month in MONTHS:
    mtag = month.split("-")[0]
    print(f"Total redeemers {month}: {out[f'Total ({mtag})'].sum()}")
display(spark.createDataFrame(out))

In [0]:
# ============================================================
# REDEMPTION SUMMARY — Oct+Nov 2026 combined (summed, not de-duped)
# A member active in both months counts once per month = 2 participants.
# All figures from df_redemption_proj (trend + seasonal fit).
# ============================================================
import pandas as pd

_pn = df_redemption_proj[df_redemption_proj["Period"].isin(["Oct-2026","Nov-2026"])]

rows = []
for brand in ["Rec Room", "Playdium"]:
    b = _pn[_pn["Brand"] == brand]
    members   = b["Proj_DistinctMembers"].sum()   # Oct + Nov
    redeemers = b["Proj_Redeemers"].sum()          # Oct + Nov
    rows.append({
        "Brand": brand,
        "Projected Members": round(members),
        "Redeemers": round(redeemers),
        "Redemption Ratio": f"{redeemers/members*100:.2f}%",
        "Members per 1pp": round(members/100),
    })

summary = pd.DataFrame(rows)
tm, tr = summary["Projected Members"].sum(), summary["Redeemers"].sum()
summary = pd.concat([summary, pd.DataFrame([{
    "Brand": "Total", "Projected Members": tm, "Redeemers": tr,
    "Redemption Ratio": f"{tr/tm*100:.2f}%", "Members per 1pp": round(tm/100),
}])], ignore_index=True)

display(spark.createDataFrame(summary))